# 07 - Writers: run, store to HDF5, reopen and plot

This notebook runs a tiny ShakerMaker model, stores the station responses with
`HDF5StationListWriter`, then reopens the `.h5` file with `h5py` and plots one
station's velocity / acceleration / displacement traces.

Every figure is saved as a `.png` in this folder.

## Build a small model

Two surface stations over the SCEC LOH.1 crust, a single Gaussian point
source at 2 km depth. We keep `nfft=2048` so the run is fast.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from shakermaker.shakermaker import ShakerMaker
from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.slw_extensions import HDF5StationListWriter

dt, nfft, tb, dk, tmax = 0.025, 2048, 1000, 0.1, 30

crust = SCEC_LOH_1()

sigma = 0.06
t0 = 6 * sigma
M0 = 1e18 / 5e14 / 2
stf = Gaussian(t0=t0, freq=1 / sigma, M0=M0, derivative=False)
source = PointSource([0, 0, 2], [0, 90, 0], stf=stf)
fault = FaultSource([source], metadata={"name": "src"})

s1 = Station([6, 8, 0], metadata={"name": "sta01"})
s2 = Station([8, 8, 0], metadata={"name": "sta02"})
stations = StationList([s1, s2], {})
model = ShakerMaker(crust, fault, stations)

## Preview the model before running
Before executing, we inspect the crust (layered column + velocity profile), the source geometry, and its source time function (STF). Each figure is saved as a PNG in this folder.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.tools.plotting import SourcePlot

crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## Check parameters and run with the HDF5 writer

`check_parameters` is pure arithmetic; call it before running. The writer
stores velocity, acceleration and displacement under `/Data`.

In [ ]:
model.check_parameters(dt=dt, nfft=nfft, dk=dk, tb=tb, tmax=tmax)

outfile = "writers_demo.h5"
writer = HDF5StationListWriter(outfile)
model.run(dt=dt, nfft=nfft, tb=tb, dk=dk, tmax=tmax,
          writer=writer, writer_mode="progressive")

## Reopen the file with h5py

Walk the file and print every dataset's name and shape. Signals are stored
row-major as `3 * nstations` rows (E, N, Z per station).

In [ ]:
def walk(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  DATASET {name}  shape={obj.shape}")
    else:
        print(f"GROUP   {name}")

with h5py.File(outfile, "r") as hf:
    hf.visititems(walk)
    dt_file = float(hf["Metadata/dt"][()])
print("dt =", dt_file)

## Plot the traces of station 0

Rows `0,1,2` are the E, N, Z components of station 0. We plot velocity,
acceleration and displacement.

In [ ]:
with h5py.File(outfile, "r") as hf:
    vel = hf["Data/velocity"][0:3, :]
    acc = hf["Data/acceleration"][0:3, :]
    dis = hf["Data/displacement"][0:3, :]
    n = vel.shape[1]
t = np.arange(n) * dt_file

labels = ["E", "N", "Z"]
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for arr, ax, title in [(vel, axes[0], "velocity"),
                       (acc, axes[1], "acceleration"),
                       (dis, axes[2], "displacement")]:
    for k in range(3):
        ax.plot(t, arr[k], label=labels[k])
    ax.set_ylabel(title)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("time (s)")
axes[0].set_title("Station 0 - reopened from HDF5")
plt.gcf().savefig("writers_station0.png", dpi=150, bbox_inches="tight")
plt.show()